# Fault-line DP + Basin Hopping

Four steps beyond merging public notebooks:

1. **Couzo's certified register.** Francisco Couzo keeps exact (rational) packings for n = 1 … 10,000 at
   [github.com/franciscouzo/square-packing](https://github.com/franciscouzo/square-packing) (MIT), many of them
   records above n = 324 that no public notebook had. We convert them and keep each one only if the official metric
   certifies it after a float64 round trip.
2. **Search.** Basin hopping with Evan Daniel's Rust engine from
   [github.com/evand/square-packing](https://github.com/evand/square-packing) (`search/packer`, MIT), started from
   our current packings of the puzzles above 324 that are still plain 0°/45° constructions.
3. **Fault-line gadget DP.** Our rebuild of Chris Deotte's composition: crops of the best packings (including the hop
   results) become gadgets along a fault between two shifted lattices, so one local improvement is reused in many n.
4. **Gadget quench.** Squeeze the gadgets themselves with Evan Daniel's smooth solver `fq` (same repository): loosen
   the gadget's box a little, nudge the squares, squeeze it again. A smaller gadget lowers every puzzle that uses it.

Builds on *Thinking Outside the Grid* (the problem, the float64 trap, the 45° family) and
*Certified Merge of Public Packings* (whose output is the starting point here).

In [ ]:
import os, sys, glob, math, shutil, subprocess, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
SRC = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/hop_search.py', recursive=True) + ['../src/hop_search.py'] if os.path.exists(p))
sys.path.insert(0, SRC)
import metric
from ensemble import side, certify
KAGGLE = os.path.exists('/kaggle/input')
OUT = '/kaggle/working' if KAGGLE else '../submissions'
# starting point: the output of the public 'Certified Merge of Public Packings' notebook (locally: the same file)
BASE = next(p for p in glob.glob('/kaggle/input/**/submission.csv', recursive=True) if 'certified-merge' in p) if KAGGLE     else '../submissions/sub_ens2.csv'
COUZO, HOP = f'{SRC}/couzo.csv' if KAGGLE else '../submissions/couzo.csv', f'{SRC}/hop.csv' if KAGGLE else '../submissions/hop.csv'
BIG = f'{SRC}/couzo_big.csv' if KAGGLE else '../submissions/couzo_big.csv'
def load(path):
    df = pd.read_csv(path)
    return {int(n): g.sort_values('square_id')[['x', 'y', 'angle']].to_numpy(float) for n, g in df.groupby('n') if len(g) == n}
base = load(BASE)
print('starting point:', BASE, f'| side-based total {sum(side(base[n]) / math.sqrt(n) - 1 for n in base):.6f}')

In [ ]:
# drawing helpers used in every section: blue = upright square, orange = tilted square, red = the box
def corners(x, y, a):
    c, s = math.cos(a), math.sin(a)
    return [(x + dx * c - dy * s, y + dx * s + dy * c) for dx, dy in [(.5, .5), (-.5, .5), (-.5, -.5), (.5, -.5)]]
def draw(ax, sq, title):
    pts = [p for q in sq for p in corners(*q)]
    xs, ys = zip(*pts); L = max(max(xs) - min(xs), max(ys) - min(ys))
    ax.add_patch(Rectangle((min(xs), min(ys)), L, L, fill=False, ec='crimson', lw=2))
    for x, y, a in sq:
        tilt = abs(math.sin(2 * a)) > 1e-6
        ax.add_patch(Polygon(corners(x, y, a), fc='#f4a63a' if tilt else '#4c7fb0', ec='#222', lw=.3))
    ax.set_title(f'{title}\nn={len(sq)}  L={L:.6f}', fontsize=9); ax.set_aspect('equal'); ax.axis('off')
    ax.set_xlim(min(xs) - .3, min(xs) + L + .3); ax.set_ylim(min(ys) - .3, min(ys) + L + .3)
print('helpers ready')

## 0. Every known method on one page

The score is the sum over n = 1 … 1000 of L_n/√n − 1, so every puzzle counts, and a gain on a big n is worth as much
as the same relative gain on a small one. These are the methods that move it, roughly in the order they were found.
The column *here* says where this notebook uses each one.

| method | idea in one line | from | here |
|---|---|---|---|
| grid | ⌈√n⌉ × ⌈√n⌉ upright squares; the fallback for every puzzle | — | baseline |
| tilt a square (Göbel 1979) | n = k² + 1: four corner blocks plus squares turned 45° in the middle beat the next grid size | Göbel, Ellsworth's enumerator | starting point |
| record tables | best-known packings for all n ≤ 324 and some larger | Friedman, Ellsworth, Couzo | §1 |
| merge | take the smallest certified box per n from every public notebook | everyone | §3 |
| closures | delete squares from a smaller-box packing of n + k (L_n ≤ L_{n+k}); add an L of squares to n − m | standard | §3 |
| basin hopping | fix the box, minimise overlap with L-BFGS, kick, accept by Metropolis, shrink when overlap-free | Evan Daniel's `packer` | §2 |
| contact LP reclosure | freeze angles, choose a separating axis per touching pair, solve one LP for all centres and the box | Kirill, kozykappa, lightningv08 | inputs |
| fault-line gadget DP | cut good packings into w × h gadgets and chain them along a seam between two shifted lattices | Chris Deotte | §4 |
| gadget quench | squeeze the gadgets the DP uses most, then rerun the DP | this notebook, after Deotte's GPU squeeze | §4b |
| L-cuts | remove a strip from a bigger gadget and re-squeeze it into a smaller key | Chris Deotte (GPU) | not here |

And one trap that silently costs points: the metric certifies with interval arithmetic, so squares that touch *exactly*
(any construction with √2 in it) fail. Every packing we build is scaled by 1 + 1e-12 about its centre before it is
written, which turns contacts into gaps that survive even a float64 round trip.

**Correction 2026-10-31:** the board scores the float64 values of the CSV, not its text. From 2026-10-29 our merge
certified and wrote 45-digit text; four submissions that certified 22.2605-22.2616 as text scored 22.6798, 25.1903,
22.6309 and 25.1809 on the board, and the float64 round trip predicts all four to 7 decimals. `ensemble.py` now
certifies every packing after float64 and writes float text.

In [ ]:
# three pictures: why tilting wins at n = 5, what a closure is, and the shape of a fault-line packing
s5 = 2 + math.sqrt(2) / 2
goebel5 = [(.5, .5, 0), (s5 - .5, .5, 0), (.5, s5 - .5, 0), (s5 - .5, s5 - .5, 0), (s5 / 2, s5 / 2, math.pi / 4)]
grid5 = [(i + .5, j + .5, 0) for i in range(3) for j in range(3)][:5]
fig, ax = plt.subplots(1, 4, figsize=(18, 4.8))
draw(ax[0], grid5, 'grid: 5 squares need a 3 x 3 box'); draw(ax[1], goebel5, 'Goebel: tilt one, box 2.707')
n_c = next(n for n in range(30, 1000) if n + 1 in base and abs(side(base[n + 1]) - side(base.get(n, base[n + 1]))) < 1e-9
           and any(abs(math.sin(2 * q[2])) > 1e-6 for q in base[n + 1]))
draw(ax[2], base[n_c + 1], f'closure: a packing of {n_c + 1} ...')
xs = [p[0] for q in base[n_c + 1] for p in corners(*q)]; ys = [p[1] for q in base[n_c + 1] for p in corners(*q)]
far = max(range(n_c + 1), key=lambda i: base[n_c + 1][i][0] + base[n_c + 1][i][1])          # any square can go; take a corner one
ax[2].add_patch(Polygon(corners(*base[n_c + 1][far]), fc='#bbbbbb', ec='crimson', lw=1.5, hatch='//'))
ax[2].set_title(f'closure: delete the hatched square from {n_c + 1}\n-> a packing of {n_c} in the same box', fontsize=9)
# fault line exactly as fault_dp.compose lays it out: three 2 x 2 gadgets, K = 6, overhang F
K, F0, gad = 6, 0.55, [(2, 2), (2, 2), (2, 2)]
lo, hi, ox, oy, boxes = [K] * K, [K] * K, 0, K, []
for w, h in gad:
    oy -= h; lo[ox:ox + w] = [oy] * w; hi[ox:ox + w] = [oy + h] * w; boxes.append((ox, oy, w, h)); ox += w
for i in range(K):
    for j in range(lo[i]):
        ax[3].add_patch(Rectangle((i, j), 1, 1, fc='#4c7fb0', ec='#222', lw=.4))
    for j in range(hi[i], K):
        ax[3].add_patch(Rectangle((i + F0, j + F0), 1, 1, fc='#7fb07f', ec='#222', lw=.4))
for k, (gx, gy, w, h) in enumerate(boxes):
    ax[3].add_patch(Rectangle((gx, gy), w + F0, h + F0, fc='#f4a63a', ec='#222', lw=.8, alpha=.85))
    ax[3].text(gx + (w + F0) / 2, gy + (h + F0) / 2, f'gadget {k + 1}\n{w}x{h}+dm', ha='center', va='center', fontsize=8)
ax[3].add_patch(Rectangle((0, 0), K + F0, K + F0, fill=False, ec='crimson', lw=2))
ax[3].set_xlim(-.3, K + F0 + .3); ax[3].set_ylim(-.3, K + F0 + .3); ax[3].set_aspect('equal'); ax[3].axis('off')
ax[3].set_title('fault line: blue lattice, green lattice shifted by f,\ngadgets (w+f) x (h+f) along the seam; box K + f', fontsize=9)
plt.tight_layout(); plt.show()

## 1. Couzo's witnesses

Each `.cert` file is an exact packing: the first line is `n side`, then one line per square `x y t` as rationals,
with the rotation stored as `t = tan(θ/2)` so that cos and sin stay rational. Exact packings touch exactly, which the
official metric cannot certify after float64 rounding, so `couzo.py` converts them with 40-digit arithmetic and scales
each packing by 1 + 1e-12 about its centre. Converted file: `couzo.csv` in the starter-kit dataset.

In [ ]:
couzo = load(COUZO)
rows = []
for n, sq in couzo.items():
    if n in base and side(sq) < side(base[n]) - 1e-9:
        rows.append((n, side(base[n]), side(sq), (side(base[n]) - side(sq)) / math.sqrt(n)))
imp = pd.DataFrame(rows, columns=['n', 'side before', 'Couzo side', 'score gain'])
print(f'{len(couzo)} witnesses with n <= 1000; smaller box than the starting point at {len(imp)} puzzles, '
      f'total score gain {imp["score gain"].sum():.6f} (before certification)')
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(imp.n, imp['score gain'], width=3, color='#4c7fb0'); ax.set_xlabel('n'); ax.set_ylabel('score gain')
ax.set_title("where Couzo's register beats the merged public packings"); plt.tight_layout(); plt.show()
imp.sort_values('score gain', ascending=False).head(8)

## 2. Basin hopping

`packer hop` keeps the box side fixed and minimises the total overlap (separating-axis penetration, as in the metric)
with L-BFGS. Every step it perturbs a few squares (shift, rotate, swap) and accepts the result by a Metropolis rule.
Whenever the packing becomes overlap-free it **squeezes** the box and sets the next target just below the best side
found, so a run only ever goes down.

Starting points matter: the 0°/45° constructions are rigid and symmetric, and a few minutes of hopping often finds
a slightly tilted, slightly smaller relative. We build the engine here (needs *internet on*) and run a short live
demo; the bigger runs were done offline with the same code (`hop_search.py`), results in `hop.csv`.

In [ ]:
if KAGGLE:
    t = time.time()
    sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True)
    sh('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal')
    sh('git clone -q --depth 1 --filter=blob:none --sparse https://github.com/evand/square-packing.git /kaggle/working/evand && '
       'cd /kaggle/working/evand && git sparse-checkout set --no-cone /search/packer/src/ /search/packer/Cargo.toml /search/packer/Cargo.lock')
    r = sh('cd /kaggle/working/evand/search/packer && $HOME/.cargo/bin/cargo build --release --bin packer --bin fq')
    os.environ['PACKER_BIN'] = '/kaggle/working/evand/search/packer/target/release/packer'
    os.environ['FQ_BIN'] = '/kaggle/working/evand/search/packer/target/release/fq'          # used in section 4b
    print(r.stderr[-400:], f'built in {time.time() - t:.0f}s')
    assert os.path.exists(os.environ['PACKER_BIN']) and os.path.exists(os.environ['FQ_BIN']), 'packer/fq build failed (is internet on?)'
else:
    os.environ['PACKER_BIN'] = os.path.abspath('../external/evand/search/packer/target/release/packer')
    os.environ['FQ_BIN'] = os.path.abspath('../external/evand/search/packer/target/release/fq')
os.environ['HOP_WORK'] = os.path.abspath(f'{OUT}/hop_demo'); os.makedirs(os.environ['HOP_WORK'], exist_ok=True)
import hop_search
print('engine:', os.environ['PACKER_BIN'], os.path.exists(os.environ['PACKER_BIN']))

In [ ]:
demo = [494, 845, 903]
cur = {n: base[n] for n in demo}
t = time.time()
from multiprocessing import Pool
with Pool(3) as pool:
    res = pool.map(hop_search.run, [(n, cur[n], 3.0, 1) for n in demo])
for n, L0, L1, why, sec in res:
    print(f'n={n}: start {L0:.6f} -> {L1 if L1 is None else round(L1, 6)}  {why}  ({sec:.0f}s)')
print(f'live demo: {time.time() - t:.0f}s')

In [ ]:
hop = load(HOP)
n0 = max(hop, key=lambda n: (side(base[n]) - side(hop[n])) / math.sqrt(n))
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
draw(ax[0], base[n0], 'before: 0/45-degree construction'); draw(ax[1], hop[n0], 'after basin hopping (certified)')
plt.tight_layout(); plt.show()
print(f'{len(hop)} puzzles improved by the offline runs')

## 3. Merge everything

For every n, take the smallest box that the official metric certifies, from every attached public output and our own files.
Three rules keep this merge honest; the last two were added on 2026-10-29, after a run that silently took nothing new:

1. **Certify against the current metric.** The organisers updated it in late October (an AABB pad, coordinates measured
   relative to square 0, bounds on coordinates and angles, at most 100 characters per number). Our copy was from
   2026-10-21; it is now pulled again from `metric/squares-in-squares` and shipped in the starter kit.
2. **Certify each packing after a float64 round trip, and write the float text.** The board reads the CSV as float64.
   A 45-digit packing whose contacts certify only at full precision scores as overlapping (the grid) on the board: we wrote
   such text from 2026-10-29 to 2026-10-31, and four submissions that certified 22.2605-22.2616 as text scored 22.63-25.19.
3. **Ignore puzzles outside n = 1 … 1000.** One public merge output (nilanjansaha123, *Certified Portfolio Merge + Proj Quench*)
   now also ships packings for n up to 3,427. The metric scores only 1 … 1000, so the merge skips the rest instead of crashing
   (it stopped our 10:00 CET run on 2026-10-29 with `KeyError: 1128`).


In [ ]:
srcs = {'base': BASE, 'couzo': COUZO, 'hop': HOP}
QUENCH = f'{SRC}/quench.csv' if KAGGLE else '../kernels/starter-kit/quench.csv'   # 2026-10-24 target-gadget quench, certified
if os.path.exists(QUENCH):
    srcs['quench'] = QUENCH
for p in glob.glob('/kaggle/input/**/submission.csv', recursive=True):   # other attached public notebooks' outputs
    if p != BASE and 'starter-kit' not in p:
        srcs[p.split('/')[-2][:40]] = p
for f in glob.glob(f"{os.environ['HOP_WORK']}/[0-9]*.csv"):
    srcs['live_' + os.path.basename(f)] = f
args = [sys.executable, '-u', f'{SRC}/ensemble.py', f'{OUT}/merged.csv'] + [f'{k}={v}' for k, v in srcs.items()]
print(subprocess.run(args, capture_output=True, text=True).stdout)

## 4. Fault-line gadget DP

Idea from Chris Deotte's *multi-gadget fault DP* notebook (his markdown describes it; his notebook ships the packings, not the code,
so `fault_dp.py` is our rebuild). Write n = K² + D. Fill the lower-left of a box with upright unit cells on the integer lattice
and the upper-right with the same lattice shifted by f in x and y. The two "continents" meet along a staircase fault running
down the anti-diagonal. Along the fault sit **gadgets**: w × h blocks that hold w·h + dm squares inside (w+f) × (h+f).
Every gadget adds dm squares; the box side is K + f, with f the largest overhang in the chain.

* **Gadgets are crops.** Take any good packing (ours, Couzo's up to n = 3500, the hop results), under each of its 8 symmetries,
  and for every (w, h, dm) keep the w·h + dm squares that stick out least past w × h: that order statistic is f.
* **Seams.** Two gadgets meet corner to corner (only if the tips cannot collide), or with one lattice column or row between them.
* **DP** over (columns used, rows used, surplus, tip type of the last gadget), minimising the largest f.
  A gadget that basin hopping improved in one puzzle is then reused in every band that can hold it.
* Each composition is certified with the official metric after the float64 round trip before it is used.

In [ ]:
t = time.time()
cur, first, total = f'{OUT}/merged.csv', {}, 0
for p in range(3):          # each pass: DP on the current file (its compositions become donors), merge, closures
    r = subprocess.run([sys.executable, '-u', f'{SRC}/fault_dp.py', cur, f'{OUT}/fault{p}', COUZO, HOP, BIG],
                       capture_output=True, text=True)
    fault = {int(os.path.basename(f)[:-4]): f for f in glob.glob(f'{OUT}/fault{p}/[0-9]*.csv')}
    print(f'pass {p}: ' + next((l for l in r.stdout.splitlines() if 'DONE' in l), r.stderr[-500:]) + f' on {len(fault)} puzzles')
    if p == 0:
        first = fault
    if not fault:
        break
    args = [sys.executable, '-u', f'{SRC}/ensemble.py', f'{OUT}/m{p}.csv', f'cur={cur}'] + [f'fault_{n}={f}' for n, f in fault.items()]
    subprocess.run(args, capture_output=True, text=True)
    subprocess.run([sys.executable, '-u', f'{SRC}/closure.py', f'{OUT}/m{p}.csv', f'{OUT}/c{p}.csv'], capture_output=True, text=True)
    cur = f'{OUT}/c{p}.csv'
print(f'({time.time() - t:.0f}s)')
if first:
    merged = load(f'{OUT}/merged.csv')
    n1 = max(first, key=lambda n: (side(merged[n]) - side(load(first[n])[n])) / math.sqrt(n))
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    draw(ax[0], merged[n1], 'before'); draw(ax[1], load(first[n1])[n1], 'fault-line composition (certified)')
    plt.tight_layout(); plt.show()

## 4b. Gadget quench

The DP only rearranges gadgets it already has, so its result is limited by how tight each gadget is.
Here we make the gadgets themselves tighter.

**Quench** is a squeeze. Take a gadget (say 7 × 17 cells holding 130 squares, sticking out by f = 0.90174),
make its box 0-4 % looser, nudge every square a little (about 0.01 in position, a fraction of a degree in angle), then
shrink the box as far as it goes without any overlap. Evan Daniel's `fq` does the shrinking: instead of pushing
overlapping squares apart, it gives every touching pair its own separating line and moves the lines too, which keeps
the problem smooth, so the optimiser does not stall at flat contacts. We repeat this from the best layout so far for
60 seconds per gadget and keep the best.

**Which gadgets.** Not all gadgets move. One built from an exact construction (f = (p + q√2)/r, e.g. Göbel's 8 × 8 + 1
at f = 0.5355) is rigid: every square is locked by contacts and every squeeze returns it unchanged. One found by a
numerical search is usually *almost* rigid: a few squares have a little slack, and a squeeze can win something.
So we target the 78 gadgets that the DP leans on most (ranked offline: lower one gadget's f by 0.005, rerun the DP, sum
the gains over all puzzles) and whose f is not such an exact value. Measured on our best file: 21 of them gave
15 smaller gadgets and +0.0015 over 29 puzzles; most of it came from long thin ones (7 × 16, 7 × 19, 5 × 18, 8 × 23).

Every improved gadget joins the DP's table; every new composition is certified with the official metric before it is used.

In [ ]:
t = time.time()
KEYS = f'{SRC}/quench_keys.json'
r = subprocess.run([sys.executable, '-u', f'{SRC}/quench_gadgets.py', cur, f'{OUT}/quench', KEYS, '60', str(os.cpu_count() or 4),
                    COUZO, HOP, BIG], capture_output=True, text=True, env=dict(os.environ, OMP_NUM_THREADS='1'))
lines = r.stdout.splitlines()
moved = [l.split('] ', 1)[1] for l in lines if 'IMPROVED' in l]
print(f'{len(moved)} gadgets got smaller:'); print('\n'.join(moved[:12]))
print(next((l for l in lines if 'DONE' in l), r.stderr[-800:]), f'({time.time() - t:.0f}s)')
qf = {int(os.path.basename(f)[:-4]): f for f in glob.glob(f'{OUT}/quench/[0-9]*.csv')}
if qf:
    args = [sys.executable, '-u', f'{SRC}/ensemble.py', f'{OUT}/mq.csv', f'cur={cur}'] + [f'quench_{n}={f}' for n, f in qf.items()]
    subprocess.run(args, capture_output=True, text=True)
    subprocess.run([sys.executable, '-u', f'{SRC}/closure.py', f'{OUT}/mq.csv', f'{OUT}/cq.csv'], capture_output=True, text=True)
    cur = f'{OUT}/cq.csv'

In [ ]:
# one gadget before and after: the largest drop in f
import json
def draw_gadget(ax, sq, w, h, title):
    pts = [p for q in sq for p in corners(*q)]
    x0, y0 = min(p[0] for p in pts), min(p[1] for p in pts)
    f = max(max(p[0] for p in pts) - x0 - w, max(p[1] for p in pts) - y0 - h, 0)
    ax.add_patch(Rectangle((x0, y0), w, h, fill=False, ec='#999', lw=1, ls='--'))
    ax.add_patch(Rectangle((x0, y0), w + f, h + f, fill=False, ec='crimson', lw=2))
    for x, y, a in sq:
        ax.add_patch(Polygon(corners(x, y, a), fc='#f4a63a' if abs(math.sin(2 * a)) > 1e-6 else '#4c7fb0', ec='#222', lw=.3))
    ax.set_title(f'{title}\n{w} x {h} + {len(sq) - w * h}   f = {f:.6f}', fontsize=9); ax.set_aspect('equal'); ax.axis('off')
    ax.set_xlim(x0 - .3, x0 + w + f + .3); ax.set_ylim(y0 - .3, y0 + h + f + .3)
gq = np.load(f'{OUT}/quench/gadgets_quenched.npz') if os.path.exists(f'{OUT}/quench/gadgets_quenched.npz') else {}
drops = []
for l in lines:
    if 'IMPROVED' in l:
        k, rest = l.split('] ', 1)[1].split(': f ')
        f0, f1 = (float(v) for v in rest.split(' (')[0].split(' -> '))
        drops.append((f0 - f1, k))
if drops:
    _, k = max(drops)
    w, h, dm = (int(v) for v in k.replace('+', 'x').split('x'))
    fig, ax = plt.subplots(1, 2, figsize=(9, 9) if h > w else (14, 5))
    draw_gadget(ax[0], gq[f'{w}_{h}_{dm}_before'], w, h, 'before (dashed: w x h cells, red: box)')
    draw_gadget(ax[1], gq[f'{w}_{h}_{dm}'], w, h, 'after quench')
    plt.tight_layout(); plt.show()

## 5. Official score

In [ ]:
shutil.copy(cur, f'{OUT}/submission.csv')
t = time.time()
s = pd.read_csv(f'{OUT}/submission.csv', float_precision='round_trip')   # float64, correctly rounded, as the board reads it
print(f"official metric, full file after float64 (as the board reads it): {metric.score(pd.DataFrame({'id': range(1000), 'n': range(1, 1001)}), s, 'id'):.6f} ({time.time() - t:.0f}s)")

## 6. What we learned

* **Search rate.** A pilot of 24 construction puzzles × 5 CPU-minutes improved 4 of them, +0.000297 in total:
  about 1.5e-4 score per CPU-hour. Small per run, but there are ~220 such puzzles above n = 324 and nothing else is
  searching them.
* **The grid puzzles are not the target.** 457 puzzles are still the plain grid; for none of them is any smaller box
  known (checked against Couzo's register), and 144 are proven optimal (n = k², k² − 1 … k² − 4). They add the same
  ~7.36 to every team's score.
* **Composition multiplies search.** The fault DP turned hop results on single puzzles (e.g. n = 372, 451) into gadgets
  for larger puzzles: 47 more puzzles improved, +0.0021, from donors we already had. Re-running it on its own output
  gives +0.0002, then +0.0001: it converges in about three passes.
* **Gadgets beat puzzles.** Squeezing 21 gadgets for 60 s each gave +0.0015, about 0.004 score per CPU-hour:
  30-50 times what basin hopping on whole puzzles gives, because each gadget is reused across many puzzles.
* **Rigid gadgets do not move.** The biggest levers (8 × 8 + 1, 12 × 12 + 4, 16 × 16 + 9) are exact constructions;
  a squeeze returns them unchanged. Improving those needs a new construction, not more search.
* **What decides the leaderboard**: syncing new public records quickly (ties go to the earliest submission) and
  search on the non-grid puzzles above 324.

**What did not work** (measured on our files; each is a null, so you can skip it):

| tried | result |
|---|---|
| quench on exact-construction gadgets (8 × 8 + 1, 12 × 12 + 4, 16 × 16 + 9) | 0: they are rigid, every squeeze returns them unchanged |
| "cushion shift": move a gadget with f < F up or right inside its slot so more seams need no buffer | 0 gain at five values of F; chains above n = 324 have room for buffer seams |
| synthetic tilted-block gadgets (a turned a × b block in an upright grid) | 0.14 to 0.29 worse in f than cropped gadgets |
| a local LLM (30B coder) writing gadget constructors, then gadget searches | 0 new gadgets in about 1,600 programs; the plain squeeze loop it was meant to beat found them |

**Public notebooks whose outputs this notebook merges** (their packings are most of the score; thank you):

* [Кирилл Тымчишин (kirilltym)](https://www.kaggle.com/kirilltym): the contact-LP reclosure series (separation-graph, seam LP,
  donor LP, wide LP): fixed-angle LPs over separating axes that move whole rows of squares at once.
* [Vladislav Lavrentev (lightningv08)](https://www.kaggle.com/code/lightningv08/squares-in-squares-cuda-packing-search):
  CUDA packing search, deletion starts from neighbouring n, and the Apache-2.0 contact-polish code many notebooks build on.
* [Chris Deotte (cdeotte)](https://www.kaggle.com/cdeotte): multi-gadget fault DP, L-cut gadgets and GPU squeezing, and the ensemble.
* [Malyshev Danil (malyshevdanil)](https://www.kaggle.com/malyshevdanil): fresh certified records and neighbour transfer + squeeze.
* [busyaprime](https://www.kaggle.com/busyaprime): [*Public Packings + fq Quench*](https://www.kaggle.com/code/busyaprime/33-lb-22-2683156-public-packings-fq-quench); 513 of the puzzles in the 2026-10-28 merge (22.266616) are theirs.
* [kozykappa](https://www.kaggle.com/code/kozykappa/subset-transfer-contact-refinement-and-compaction): subset transfer, contact refinement and compaction.
* [SeshuRaju (seshurajup)](https://www.kaggle.com/seshurajup), [yuki16](https://www.kaggle.com/yuki16),
  [aqibrazadev](https://www.kaggle.com/aqibrazadev), [dariushafshar](https://www.kaggle.com/dariushafshar): ensembles, merges and validators.

*Credits: Chris Deotte (the fault-line gadget idea, and ranking gadgets by DP leverage), Francisco Couzo (certified register), Evan Daniel and contributors (evand/square-packing, packer and fq engines),
David Ellsworth (records, enumerator), Erich Friedman's survey, and the public notebooks merged in the starting point.
If this helped, an upvote is appreciated.*